# Compact message-boundary layouts on reconstructed cons_instr_sink, plan4_dev, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

`plan4_dev` only. Do **not** load `plan4_hold` or `plan4_fresh`. Do not resume recon, iface, support, or hold.

A is not regenerated. Frozen compact IDs come from `results_stage_plan4_iface/a_artifacts.jsonl`. Result wording, B's instructions, decoding, and scoring stay unchanged.

The reconstructed compact prefix currently begins `<|im_start|>user\nYouP: ...` and has no end-of-turn token before B's new user turn.

Four reconstructed layouts:

1. **current** — frozen compact IDs, unchanged.
2. **sep** — insert a newline between the first four tokens and the result lines.
3. **close** — close the preceding message with the chat end-of-turn tokens.
4. **both** — separator and turn closure.

Inspect and the four reconstructed layouts are done. `sep` is the primary layout: 89/90 labels, one added newline token. `both` matched those labels but is larger.

**Next: cells 7–8**, `--phase transfer --layout sep` (180 evaluations). This does not rewrite the 360 reconstructed rows.

- Transferred KV + separator: keep A’s initial and result states, forward the newline, RoPE the retained keys.
- Reconstructed KV + separator: rebuild the identical token sequence.

Do not load `plan4_hold`. Do not train an adapter yet.

Download `/kaggle/working/results_stage_plan4_boundary`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_boundary.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_boundary.py").is_file() and (cand / "data" / "plan4_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_boundary.py and data/plan4_dev.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_boundary.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan4_dev", sum(1 for _ in (REPO / "data" / "plan4_dev.jsonl").open(encoding="utf-8")))
assert not (REPO / "data" / "plan4_hold.jsonl").samefile(REPO / "data" / "plan4_dev.jsonl")
art = REPO / "results_stage_plan4_iface" / "a_artifacts.jsonl"
print("frozen A artifacts", art.is_file(), art)

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_boundary")
print("Run inspect first. Only then reconstruct. Do not run transfer yet.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_boundary", "--split", "plan4_dev", "--phase", "inspect"])
print("plan4 boundary inspect exit", rc)
path = WORK / "results_stage_plan4_boundary" / "inspect.json"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_boundary" / "run" / "inspect.json"
if path.is_file():
    data = json.loads(path.read_text(encoding="utf-8"))
    slim = {k: data.get(k) for k in ("n", "all_passed", "sep_text", "close_text", "lead")}
    print(json.dumps(slim, indent=2)[:8000])
rows_path = path.parent / "inspect.jsonl"
if rows_path.is_file():
    rows = [json.loads(line) for line in rows_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    key = rows[0]["a_state_key"]
    for layout in ("current", "sep", "close", "both"):
        row = next(r for r in rows if r["a_state_key"] == key and r["layout"] == layout)
        print("\n====", layout, key, "====")
        print(row["prefix_text"][:500])
        print("--- B turn head ---")
        print(row["full_text"][len(row["prefix_text"]):len(row["prefix_text"])+180])

In [ ]:
src = WORK / "results_stage_plan4_boundary" / "inspect.json"
if not src.is_file():
    src = REPO / "results" / "stage_plan4_boundary" / "run" / "inspect.json"
inspect = json.loads(src.read_text(encoding="utf-8"))
if not inspect.get("all_passed"):
    raise RuntimeError("Inspect failed. Do not run the 90-case reconstructed comparison yet.")
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_boundary", "--split", "plan4_dev", "--phase", "reconstruct"])
print("plan4 boundary reconstruct exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_boundary"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_boundary" / "run"
print("reading", src)
path = src / "summary.json"
if path.is_file():
    data = json.loads(path.read_text(encoding="utf-8"))
    keep = {k: data.get(k) for k in ("n_rows", "n_examples", "receiver", "carrier_label_errors", "family_clustered_eligibility", "qualifies_vs_text", "frozen_text_result", "interpretation_guide", "payload_note", "lead")}
    print(json.dumps(keep, indent=2)[:24000])
print("Do not run transfer until these reconstructed-layout numbers are read.")

In [ ]:
src = WORK / "results_stage_plan4_boundary" / "inspect.json"
if not src.is_file():
    src = REPO / "results" / "stage_plan4_boundary" / "run" / "inspect.json"
if not src.is_file():
    src = REPO / "results_stage_plan4_boundary" / "inspect.json"
inspect = json.loads(src.read_text(encoding="utf-8"))
if not inspect.get("all_passed"):
    raise RuntimeError("Inspect did not pass. Do not run transfer.")
print("inspect", src, "all_passed", inspect.get("all_passed"))
rc = subprocess.call([
    sys.executable, "-m", "src.stage_plan4_boundary",
    "--split", "plan4_dev", "--phase", "transfer", "--layout", "sep",
])
print("plan4 boundary transfer sep exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_boundary"
if not (src / "transfer_summary.json").is_file():
    alt = REPO / "results" / "stage_plan4_boundary" / "run"
    src = alt if (alt / "transfer_summary.json").is_file() else REPO / "results_stage_plan4_boundary"
print("reading", src)
path = src / "transfer_summary.json"
if path.is_file():
    data = json.loads(path.read_text(encoding="utf-8"))
    keep = {k: data.get(k) for k in ("layout", "n_rows", "n_examples", "receiver", "carrier_label_errors", "family_clustered_eligibility", "qualifies_vs_text", "frozen_text_result", "interpretation_guide", "lead")}
    print(json.dumps(keep, indent=2)[:24000])